# 13 — Inventory Optimization

This notebook is the mathematical core of the inventory system. It uses a genuine periodic-review `(R,S)` policy.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
from src.inventory.inventory_policy import InventoryPolicy, calculate_policy
examples=[]
for rp in [1,3,7,14]:
    p=calculate_policy(InventoryPolicy(10,2,7,0.90,rp,1,500)); examples.append({"review_period":rp,"protection_period":p["protection_period_days"],"safety_stock":p["safety_stock"],"target_S":p["target_inventory_position"]})
import pandas as pd; display(pd.DataFrame(examples))

### Mathematics

For daily-demand uncertainty:

`Protection period = L + R`

`Safety stock = z × sigma_daily × sqrt(L + R)`

`S = mean_daily_demand × (L + R) + safety_stock`

There is no second uncertainty inflation later in the simulator.

In [ ]:
from src.utils.config import load_yaml
cfg=load_yaml("inventory_config.yaml"); policies=[]
mean=10; sd=2; L=7
for sl in cfg["service_levels"]:
    for rp in cfg["review_periods"]:
        p=calculate_policy(InventoryPolicy(mean,sd,L,sl,rp,1,500)); policies.append({**p})
grid=pd.DataFrame(policies); display(grid[["service_level","review_period_days","protection_period_days","safety_stock","target_inventory_position"]].sort_values(["service_level","review_period_days"]))

### Optimization principle

The 36 candidate policies are simulated on validation data. The minimum total cost is chosen. The held-out test period is used only after that choice.

In [ ]:
import pandas as pd
from pathlib import Path
policy_path=ROOT/"data/processed/optimized_policies.parquet"
if policy_path.exists():
    selected=pd.read_parquet(policy_path)
    display(selected.head())
    print("Most common selected review period:",selected.review_period_days.mode().tolist())
    print("Most common selected service level:",selected.service_level.mode().tolist())
else:
    print("Run python -m scripts.run_inventory_simulation to produce per-series selected policies.")
